In [15]:
import numpy as np

from numba import njit

from scipy.linalg import eigh

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import fastplotlib as fpl

import optuna

from dysts.flows import Lorenz
from dysts.maps import Henon

# Init

In [63]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [64]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [65]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [66]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.shape[1] >= 3:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [67]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [68]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [69]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [70]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=15,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

In [71]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig

# Calc Init

In [593]:
@njit(fastmath=True, cache=True)
def get_spring_forces(
    connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims
):
    forces = np.zeros((num_nodes, dims))

    idx_a, idx_b = connections_list[:, 0], connections_list[:, 1]

    disp_reshaped = disp.reshape(
        num_nodes, dims
    )  # So now for every index/time_step we have a 2D array where every index is x,y,z

    pos_a = initial_pos[idx_a] + disp_reshaped[idx_a]
    pos_b = initial_pos[idx_b] + disp_reshaped[idx_b]

    r_vecs = pos_b - pos_a
    current_lens = np.sqrt(np.sum(r_vecs**2, axis=1))

    force_magnitudes = k_vals * (current_lens - rest_lens)

    safe_lens = np.where(current_lens < 1e-13, 1.0, current_lens)
    unit_dirs = r_vecs / safe_lens.reshape(-1, 1)

    forces[idx_a] += force_magnitudes[:, np.newaxis] * unit_dirs
    forces[idx_b] -= force_magnitudes[:, np.newaxis] * unit_dirs

    return forces.reshape(-1)

In [638]:
@njit(fastmath=True, cache=True)
def run_simulation(
    steps,
    dt,
    m_inv_diag,
    c_diag,
    U,
    initial_pos,
    connections_list,
    k_vals,
    wall_nodes=[-1],
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    init_vecs = nodes_pos[connections_list[:, 0]] - nodes_pos[connections_list[:, 1]]
    rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )
    F_spring *= mask

    for i in range(1, 10):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )
        F_spring *= mask

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Lattic Lorenz

In [603]:
N = 10

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

plot_grid(nodes_pos).show()

In [604]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = rng.uniform(0.1, 0.3, size=num_nodes)*3
m_diag = np.repeat(node_m, dims)
m_inv_diag = 1.0 / m_diag

node_c = rng.uniform(0.05, 0.3, size=num_nodes)*3
c_diag = np.repeat(node_c, dims)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
# target_nodes=[]
target_node_count = lorenz_scaled.shape[1] * 4
target_nodes = rng.integers(low=0, high=num_nodes, size=(target_node_count // 2,))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
vectorized_force = np.repeat(lorenz_scaled, 4, axis=1)
U[:, col_indices] = vectorized_force

In [605]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_nodes = np.concatenate(
    [
        node_ids[:, :-1].flatten(),
        node_ids[:-1, :].flatten(),
        node_ids[:-1:2, :-1].flatten(),
        node_ids[1:-1:2, 1:].flatten(),
    ]
)

dst_nodes = np.concatenate(
    [
        node_ids[:, 1:].flatten(),
        node_ids[1:, :].flatten(),
        node_ids[1::2, 1:].flatten(),
        node_ids[2::2, :-1].flatten(),
    ]
)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [606]:
np.sqrt(m_diag.min() / k_vals.max())

np.float64(0.1957705847441128)

In [607]:
displacement, velocity = run_simulation(
    steps=steps + transient_steps_reservoir + tau_steps,
    dt=0.001,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U*.3,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

X = np.column_stack((displacement, velocity))

In [609]:
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    lorenz_train_scaled[transient_steps_reservoir + tau_steps :],
    lorenz_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = lorenz_scaler.inverse_transform(Y_pred_scaled)
Y_test = lorenz_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

-0.0617 8.6303


In [610]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
    vel=velocity
).show()

# Lattic Lorenz

In [611]:
N = 10

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

plot_grid(nodes_pos).show()

In [612]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = rng.uniform(0.1, 0.3, size=num_nodes) * 3
m_diag = np.repeat(node_m, dims)
m_inv_diag = 1.0 / m_diag

node_c = rng.uniform(0.05, 0.3, size=num_nodes)*3
c_diag = np.repeat(node_c, dims)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
# target_nodes=[]
target_nodes = rng.integers(low=0, high=num_nodes, size=3)
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
vectorized_force = np.repeat(lorenz_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [613]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_nodes = np.concatenate(
    [
        node_ids[:, :-1].flatten(),
        node_ids[:-1, :].flatten(),
        node_ids[:-1:2, :-1].flatten(),
        node_ids[1:-1:2, 1:].flatten(),
    ]
)

dst_nodes = np.concatenate(
    [
        node_ids[:, 1:].flatten(),
        node_ids[1:, :].flatten(),
        node_ids[1::2, 1:].flatten(),
        node_ids[2::2, :-1].flatten(),
    ]
)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [614]:
np.sqrt(m_diag.min() / k_vals.max())

np.float64(0.1957705847441128)

In [639]:
displacement, velocity = run_simulation(
    steps=steps + transient_steps_reservoir + tau_steps,
    dt=0.001,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U*0,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

X = np.column_stack((displacement, velocity))

In [ ]:
displacement[1]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

: 

In [635]:
arr = np.array([[1.0, np.nan, 3.0], [4.0, 5.0, np.nan]])

indices = np.where(np.isnan(displacement))
print(indices)

(array([    1,     1,     1, ..., 22000, 22000, 22000], shape=(4399847,)), array([ 53,  55,  57, ..., 197, 198, 199], shape=(4399847,)))


In [633]:
displacement[indices[0]]

array([[ 0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
        

In [619]:
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    lorenz_train_scaled[transient_steps_reservoir + tau_steps :],
    lorenz_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = lorenz_scaler.inverse_transform(Y_pred_scaled)
Y_test = lorenz_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

-0.0617 8.6303


In [620]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
    vel=velocity
).show()